# A Flax Optimization Cookbook

This notebook goes through some common problems in nontrivial training loops for flax models. For clarity, all sections below will be training the following toy model. Both raw Jax and Flax code are shown for comparison.

In [1]:
import jax
from flax import nnx
jax.config.update('jax_num_cpu_devices', 8)
import jax.numpy as jnp
from jax import tree
import optax
import itertools as it
import functools as ft
from collections import namedtuple

Here is the NNX version:

In [2]:
def nnx_model(rngs):
    return nnx.Sequential(nnx.Linear(2,8, rngs=rngs), nnx.Linear(8,8, rngs=rngs))

In [3]:
model = nnx_model(nnx.Rngs(0))

In [4]:
def nnx_loss_fn(params, x, y):
    return jnp.sum((y - model(params, x))**2)

In [5]:
def nnx_loss_fn(model, x, y):
    return jnp.sum((model(x) - y) ** 2)

And here is its equivalent raw Jax representation:

In [6]:
keys = map(ft.partial(jax.random.fold_in, jax.random.key(0)), it.count())

In [7]:
param_init = jax.nn.initializers.lecun_normal()

In [18]:
def make_linear(size, keys):
    return {
        'w': param_init(next(keys), size),
        'b': jnp.zeros(size[1])
    }

In [19]:
def jax_params(keys):
    return [make_linear((2, 8), keys), make_linear((8, 8), keys)]

In [20]:
def jax_model(params, x):
    for p in params:
        x = x @ p['w'] + p['b']
    return x

In [21]:
def jax_loss_fn(params, x, y):
    return jnp.sum((y - jax_model(params, x))**2)

We'll operate on the following fake data:

In [22]:
x = jax.random.normal(next(keys), (32, 2))
y = jax.random.normal(next(keys), (32, 8))

And we'll use ADAM to update.

In [13]:
optimizer = optax.adam(1e-3)

# Exponential Moving Average

Neural network see increased robustness when, rather than using only the weights available at the end of training, we use an exponential moving average of the weights produced throughout training. It is easy to modify the standard Jax training loop to accomodate calculating exponential moving averages. 

## EMA in Pure Jax

To start, we will see how to keep track of exponential moving averages in raw Jax. Although the raw just implementation is simple and easy to understand, it does not allow for mutable state. 

In [23]:
def ema_update(ema, new_val, decay=0.9):
    return decay * ema + (1 - decay) * new_val

In [24]:
@jax.jit
def train_step(opt_state, params, ema_params, x, y):
    loss, grads = jax.value_and_grad(jax_loss_fn)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    ema_params = tree.map(ema_update, ema_params, params)
    return opt_state, params, ema_params, loss

In [25]:
params = jax_params(keys)
opt_state = optimizer.init(params)
ema_params = params

In [26]:
losses = []
for _ in range(50):
  opt_state, params, ema_params, loss = train_step(opt_state, params, ema_params, x, y)
  losses.append(loss)

## EMA in Flax

Now, we can see how to implement an exponential moving average in Flax. The code below is almost identical to the pure jax version above, but because NNX allows for mutable operations, we no longer need to explicitly pass around the full state object. 

In [39]:
model = nnx_model(nnx.Rngs(0))

In [40]:
nnx_optimizer = nnx.Optimizer(
  model,
  tx=optimizer,
  wrt=nnx.Param,
)

In [49]:
class Ema(nnx.Module):
    def __init__(self, model):
        self.ema = nnx.merge(*nnx.split(model)) # Make a copy
    def update(self, model):
        self.ema = tree.map(ema_update, self.ema, model)

In [50]:
ema = Ema(model)

In [51]:
@nnx.jit
def nnx_train_step(model, nnx_optimizer, ema, x, y):
  loss, grads = nnx.value_and_grad(nnx_loss_fn)(model, x, y)
  nnx_optimizer.update(model, grads)
  ema.update(model)
  return loss

In [52]:
losses = []
for _ in range(50):
  loss = nnx_train_step(model, nnx_optimizer, ema, x, y)
  losses.append(loss)

# Low Rank Adaptation

The pattern for adding low rank adaptation to an optimization loop is very similar to adding an exponential moving average. As before, we create a new pytree with the same structure as our model parameters, but here we store low rank additions to these parameters rather than weighted average values. 

## Lora in Jax

In [53]:
base_params = jax_params(keys)

In [54]:
def init_lora_param(a, k=2):
    if len(a.shape) == 2:
        return {'A': param_init(next(keys), (a.shape[0], k)), 'B': jnp.zeros((k, a.shape[1]))}
    else:
        return None

In [55]:
jax_lora_params = tree.map(init_lora_param, base_params)

In [56]:
opt_state = optimizer.init(jax_lora_params)

In [57]:
def apply_lora_param(base_params, lora_params):
    if lora_params is None:
        return base_params
    return base_params + (lora_params['A'] @ lora_params['B'])

In [59]:
def jax_lora_loss(lora_params, params, x, y):
    params = tree.map(apply_lora_param, params, lora_params)
    return jax_loss_fn(params, x, y)

In [60]:
@jax.jit
def lora_train_step(params, lora_params, opt_state, x, y):
    loss, grads = jax.value_and_grad(jax_lora_loss)(lora_params, params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    lora_params = optax.apply_updates(lora_params, updates)
    return params, lora_params, opt_state, loss

In [61]:
losses = []
for _ in range(50):
  params, jax_lora_params, opt_state, loss = lora_train_step(params, jax_lora_params, opt_state, x, y)
  losses.append(loss)

## LORA in Flax

If Flax, we just need to wrap the optax optimizer with `nnx.Optimizer` to provide a mutable interface. 

In [64]:
nnx_lora_params = tree.map(init_lora_param, model)

In [65]:
def nnx_lora_loss(lora_params, params, x, y):
    params = tree.map(apply_lora_param, params, lora_params)
    return nnx_loss_fn(params, x, y)

In [66]:
@nnx.jit
def nnx_lora_train_step(nnx_model, nnx_lora_params, nnx_optimizer, x, y):
  loss, grads = nnx.value_and_grad(nnx_lora_loss)(nnx_lora_params, nnx_model, x, y)
  nnx_optimizer.update(nnx_lora_params, grads)
  return loss

In [71]:
nnx_lora_optimizer = nnx.Optimizer(
  nnx_lora_params,
  tx=optimizer,
  wrt=nnx.Param,
)

In [72]:
losses = []
for _ in range(50):
  loss = nnx_lora_train_step(model, nnx_lora_params, nnx_lora_optimizer, x, y)
  losses.append(loss)

# LBFGS

## LBFGS in Jax

In [328]:
def make_lbfgs_state(lbfgs):
    params = make_params(keys)
    opt_state = lbfgs.init(params)
    return (params, opt_state)

In [329]:
@jax.jit
def train_step(x, y, params, opt_state):
    local_loss = lambda p: loss_fn(p, x, y)
    value_and_grad_fn = optax.value_and_grad_from_state(local_loss)
    loss, grad = value_and_grad_fn(params, state=opt_state)
    updates, opt_state = lbfgs.update(grad, opt_state, params,
                                      value=loss, grad=grad, value_fn=local_loss)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

In [330]:
lbfgs = optax.lbfgs()
params, opt_state = make_lbfgs_state(lbfgs)

In [331]:
losses = []
for _ in range(50):
  loss = train_step(x, y, params, opt_state)
  losses.append(loss)

## LBFGS in Flax

TODO

# Per-Parameter Learning Rates

In some training regimes, you will want to optimize different parameters with different learning rates. 

## In Jax

First, we map from each leaf to the type of parameter it is (weight or bias).

In [74]:
params = jax_params(keys)
param_tys = jax.tree.map_with_path(lambda p, _: p[-1].key, params)

Next, we create a dictionary giving the learning rates to use for each parameter type.

In [75]:
rates = {'w': optax.adam(1e-3), 'b': optax.adam(1e-2)}

Finally, we can make a compound optimizers that uses each rate appropriately. 

In [79]:
joint_optimizer = optax.partition(rates, param_tys)
opt_state = joint_optimizer.init(params)

In [80]:
@jax.jit
def train_step(opt_state, params, x, y):
    loss, grads = jax.value_and_grad(jax_loss_fn)(params, x, y)
    updates, opt_state = joint_optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return opt_state, params, loss

In [81]:
losses = []
for _ in range(50):
  loss = train_step(opt_state, params, x, y)
  losses.append(loss)

## In Flax

TODO

# Gradient Accumulation

Just wrap it in

In [ ]:
optax.MultiSteps(optimizer, every_k_schedule=3)

# Sharding Optimization State Differently from Parameters

## Jax Version

Say we're doing data parallelism. We want to replicate our parameters across all GPUs so we can do the forward and backward passes without communication latency. 

But we don't need to replicate the optimizer state, as it's not invovled in SPMD computations. One copy is enough, and we can shard this copy across our mesh to reduce memory usage. This means that we need the optimier state to be sharded differently from the parameters themselves.

In [7]:
from jax.sharding import PartitionSpec as P, AxisType, get_abstract_mesh, reshard

In [25]:
mesh = jax.make_mesh((2, 4), ("x", "y"),
                     axis_types=(AxisType.Explicit, AxisType.Explicit))
jax.set_mesh(mesh);

To do this, we can change our initializer to take a `sharding` argument. 

In [26]:
def make_params(keys, sharding):
    return {
        'w': param_init(next(keys), (2, 8), out_sharding=sharding),
        'b': jnp.zeros(5)
    }

We'll pass in in sharding when we initialize the optimizer, which will shard the optimization state the same way. But when we initialize the model parameters themselves, we won't provide a sharding, allowing for data parallelism. 

In [31]:
opt = optimizer.init(jax.eval_shape(lambda: make_params(keys, P('x', 'y'))))

## Flax Version

In [81]:
def make_model(sharding):
    return nnx.Linear(2, 8, rngs=nnx.Rngs(0), kernel_init=ft.partial(param_init, out_sharding=sharding));

In [83]:
model = make_model(P('x', 'y'))

In [67]:
ghost_model = jax.eval_shape(lambda: make_model(P('x', 'y')))

In [70]:
opt = nnx.Optimizer(ghost_model, optax.adam(1e-3), wrt=nnx.Param)

In [87]:
jax.typeof(opt.opt_state[0].mu['kernel'][...])

ShapedArray(float32[2@x,8@y])